# KnotFold

In [ ]:
import os
import pandas as pd
import time
import subprocess
import sys
from pathlib import Path
import shutil

In [ ]:
method_name = "KnotFold"

base = Path.cwd()
knotfold_dir = Path.cwd().parent / "tools" / "KnotFold"/ "KnotFold"
knotfold_env = knotfold_dir.parent / "knotfold_env"

print(f"Current directory (methods): {base}")
print(f"KnotFold directory: {knotfold_dir}")
print(f"KnotFold environment: {knotfold_env}")

In [ ]:
# Clone KnotFold if needed
if not knotfold_dir.exists():
    print("Cloning KnotFold source code...")
    os.makedirs(str(knotfold_dir.parent), exist_ok=True)
    os.chdir(str(knotfold_dir.parent))
    !git clone https://github.com/gongtiansu/KnotFold.git
    os.chdir(str(Path.cwd()))
    print("KnotFold cloned successfully")
else:
    print(f"KnotFold already exists at {knotfold_dir}")

In [ ]:
# Create conda environment if needed
if not knotfold_env.exists():
    print("Creating KnotFold conda environment...")
    cmd = f"conda create --prefix {knotfold_env} python=3.8 pytorch::pytorch pytorch::torchvision pytorch::torchaudio pytorch::pytorch-cuda=11.8 numpy lmdb click tqdm tabulate pandas scikit-learn -y -c pytorch -c conda-forge"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✓ KnotFold environment created successfully")
    else:
        print(f"Warning: Environment creation may have issues")
        if result.stderr:
            print(f"Error output: {result.stderr[:500]}")
    
    print("\n✓ KnotFold environment setup complete")
else:
    print(f"✓ KnotFold environment already exists at {knotfold_env}")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses2.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def convert_bpseq_to_dot(bpseq_file):
    """Convert BPSEQ file to dot-bracket notation using ct2dot.py"""
    try:
        # Use ct2dot.py with relative path
        ct2dot_path = Path.cwd() / "ct2dot.py"
        bpseq_path = Path(bpseq_file)
        output_dot_file = bpseq_path.parent / (bpseq_path.stem + ".dot")
        
        cmd = [
            'python',
            str(ct2dot_path),
            str(bpseq_path),         # positional: input file
            str(output_dot_file),    # positional: output file
            '-t', 'bpseq',           # specify input format
            '-f', 'simple'           # output only the structure line
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"[ERROR] ct2dot.py failed: {result.stderr}")
            return None
        
        # Read the output dot file
        if output_dot_file.exists():
            with open(output_dot_file, 'r') as f:
                dot_notation = f.read().strip()
            # Clean up the output file
            output_dot_file.unlink()
            return dot_notation
        else:
            print(f"[ERROR] Output file not created: {output_dot_file}")
            return None
            
    except Exception as e:
        print(f"[ERROR] Exception in convert_bpseq_to_dot: {str(e)}")
        return None

In [ ]:
def run_knotfold_prediction(input_fasta, output_dir, cuda=True, gpu_id=2):
    """Run KnotFold prediction with CUDA support"""
    os.makedirs(output_dir, exist_ok=True)
    
    input_fasta_abs = str(Path(input_fasta).absolute())
    output_dir_abs = str(Path(output_dir).absolute())
    knotfold_script = str(knotfold_dir / 'KnotFold.py')
    cuda_flag = "" if cuda else "--cuda"
    
    # Run from KnotFold directory to ensure relative paths work
    cmd = f"cd {knotfold_dir} && conda run --prefix {knotfold_env} python {knotfold_script} -i {input_fasta_abs} -o {output_dir_abs} {cuda_flag}"
    
    env = os.environ.copy()

    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id) if cuda else ''
    
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
        if result.returncode != 0:
            err_msg = (result.stderr or result.stdout or "").strip()
            print(f"[ERROR] KnotFold failed with return code {result.returncode}")
            if err_msg:
                print(err_msg)
            return False
        return True
    except Exception as e:
        print(f"[ERROR] Exception in run_knotfold_prediction: {str(e)}")
        return False

In [ ]:
out_fasta_name = method_name
output_dir_base = base.parent / "prediction"
os.makedirs(output_dir_base, exist_ok=True)
output_fasta = output_dir_base / (out_fasta_name + ".fasta")

print(f"Output file: {output_fasta}")

if output_fasta.exists():
    os.remove(output_fasta)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")

for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    input_fasta = f"KnotFold_input_{i}.fasta"
    with open(input_fasta, "w") as ofile:
        ofile.write(f">{vid}\n{seq}\n")

    output_dir = f"KnotFold_temp_{i}"
  
    success = run_knotfold_prediction(input_fasta, output_dir, cuda=True, gpu_id=1)
    
    elapsed_time = time.time() - start_time
    if not success:
        print(f"{elapsed_time: .1f} s [fail] - KnotFold prediction command failed")
        if os.path.exists(input_fasta):
            os.remove(input_fasta)
        shutil.rmtree(output_dir, ignore_errors=True)
        continue

    # Check if output directory was created
    if not Path(output_dir).exists():
        print(f"{elapsed_time: .1f} s [fail] - KnotFold output directory not created")
        if os.path.exists(input_fasta):
            os.remove(input_fasta)
        continue

    # Look for BPSEQ file
    bpseq_file = Path(output_dir) / f"{vid}.bpseq"
    if not bpseq_file.exists():
        bpseq_files = list(Path(output_dir).glob("*.bpseq"))
        if not bpseq_files:
            output_files = list(Path(output_dir).glob("*"))
            print(f"{elapsed_time: .1f} s [fail] - No BPSEQ file found")
            print(f"    Output dir contents: {[f.name for f in output_files]}")
            if os.path.exists(input_fasta):
                os.remove(input_fasta)
            shutil.rmtree(output_dir, ignore_errors=True)
            continue
        bpseq_file = bpseq_files[0]
        print(f"\n    Found BPSEQ: {bpseq_file.name}")
    
    # Convert BPSEQ to dot-bracket
    structure = convert_bpseq_to_dot(str(bpseq_file))
    
    if structure is None:
        print(f"{elapsed_time: .1f} s [fail] - BPSEQ to dot conversion failed")
        if os.path.exists(input_fasta):
            os.remove(input_fasta)
        shutil.rmtree(output_dir, ignore_errors=True)
        continue
    
    # Write output
    try:
        with open(output_fasta, "a") as out_f:
            out_f.write(f">{vid}\n")
            out_f.write(f"{seq}\n")
            out_f.write(f"{structure}\n")

        print(f"{elapsed_time: .1f} s")
    except Exception as e:
        print(f"{elapsed_time: .1f} s [fail] - Write error: {str(e)}")

    # Cleanup
    if os.path.exists(input_fasta):
        os.remove(input_fasta)
    shutil.rmtree(output_dir, ignore_errors=True)